# Demo 07h — Is the Gain Real?

**Class 07 · E8** (slides: `slides/07_ensembles.md`)

Demo E7 ranks the models, but a lower error on one test set may be luck. This demo asks which gains survive a test, and which test fits which question.

* **One test set, classification:** McNemar's test looks at the rows where the two models disagree. It tests the error rate, not MCC.
* **One test set, any metric:** a paired permutation test and a bootstrap interval work on MCC and on MAE.
* **Regression:** no McNemar, because no answer is simply right. We compare the sign test, Wilcoxon and the permutation test.
* **Cross-validation:** folds share training rows, so the plain t-test rejects too often. We measure that.
* **Several tables:** Wilcoxon on the gains, with Holm's correction.

Models use fixed settings here, not the tuning of E7: only their test predictions matter. Predictions are kept in `.cache/07_holdout.csv`: delete that file to fit again (several minutes on 16 cores).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from scipy import stats
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

from faa import data, demo, metrics, plotlib
from faa import galleries as ga
from faa.demo import show, slide

demo.setup("07")
SEED = 0
ALPHA = 0.05
RIVALS = ["linear", "tree", "k-NN", "SVM", "random forest", "HistGB"]

## The hold-out set

McNemar's test needs one test set that no model touched. Every model trains on the same 70 % of a table and predicts the same 30 %. Stacking is compared with six rivals, so each table has a family of six tests.

In [ ]:
TABLES = data.TABLES
CLASS_SETS = [d for d, (_, t) in TABLES.items() if t == "c"]
REG_SETS = [d for d, (_, t) in TABLES.items() if t == "r"]


def model_of(name, task, seed=SEED):
    """Fixed settings; the SVR sees a standardized target, because its C and epsilon depend on the scale of y."""
    c = task == "c"
    models = {
        "linear": ga.scaled(LogisticRegression(max_iter=2000) if c else Ridge()),
        "tree": (DecisionTreeClassifier if c else DecisionTreeRegressor)(
            max_depth=8, min_samples_leaf=5, random_state=seed
        ),
        "k-NN": ga.scaled((KNeighborsClassifier if c else KNeighborsRegressor)(n_neighbors=15, weights="distance")),
        "SVM": ga.scaled(SVC(C=10.0) if c else TransformedTargetRegressor(SVR(C=10.0), transformer=StandardScaler())),
        "random forest": (RandomForestClassifier if c else RandomForestRegressor)(n_estimators=300, random_state=seed),
        "HistGB": (HistGradientBoostingClassifier if c else HistGradientBoostingRegressor)(
            max_iter=300, learning_rate=0.05, random_state=seed
        ),
    }
    if name == "stacking":
        members = [(k.replace(" ", "_"), v) for k, v in models.items() if k != "tree"]
        cv = (StratifiedKFold if c else KFold)(5, shuffle=True, random_state=seed)
        final = LogisticRegression(max_iter=2000) if c else Ridge()
        return (StackingClassifier if c else StackingRegressor)(members, final_estimator=final, cv=cv)
    return models[name]


def fit_predict(dataset, name):
    X, y = TABLES[dataset][0]()
    task = TABLES[dataset][1]
    strata = y if task == "c" else None
    tr, te = train_test_split(np.arange(len(y)), test_size=0.3, random_state=SEED, stratify=strata)
    with threadpool_limits(1):
        pred = model_of(name, task).fit(X[tr], y[tr]).predict(X[te])
    return pl.DataFrame(
        {"dataset": dataset, "model": name, "row": te, "y": y[te].astype(float), "pred": pred.astype(float)}
    )


CACHE = plotlib.ROOT / ".cache" / "07_holdout.csv"
if CACHE.exists():
    holdout = pl.read_csv(CACHE)
else:
    done = Parallel(n_jobs=-1, batch_size=1)(delayed(fit_predict)(d, m) for d in TABLES for m in [*RIVALS, "stacking"])
    holdout = pl.concat(done)
    holdout.write_csv(CACHE)


def predictions(dataset, name):
    here = holdout.filter(pl.col("dataset") == dataset, pl.col("model") == name).sort("row")
    return here["y"].to_numpy(), here["pred"].to_numpy()


def score_of(dataset, name):
    y, p = predictions(dataset, name)
    return metrics.task_score(TABLES[dataset][1], y, p)


show(
    pl.DataFrame(
        [
            {
                "dataset": d,
                "rows": predictions(d, "stacking")[0].size,
                **{m: score_of(d, m) for m in [*RIVALS, "stacking"]},
            }
            for d in TABLES
        ]
    )
)

## McNemar's test

Keep only the rows where exactly one model is right. If the two models were equally good, each such row would fall to either side with probability one half. The exact test is a binomial test on those rows; `scipy.stats.binomtest` is enough, so no new library is needed.

In [ ]:
slide("McNemar: Only the Disagreements Count")
y, stack = predictions("spambase", "stacking")
_, forest = predictions("spambase", "random forest")
only_stack, only_forest, p = metrics.mcnemar(y, stack, forest)
both = int(np.sum((stack == y) & (forest == y)))
neither = int(np.sum((stack != y) & (forest != y)))
print(
    f"spambase, {len(y)} test rows: both right {both}, neither {neither}, only stacking {only_stack}, only forest {only_forest}, p = {p:.3f}"
)
print(f"MCC: stacking {score_of('spambase', 'stacking'):.3f}, forest {score_of('spambase', 'random forest'):.3f}")

In [ ]:
# plot e8_table
fg, ax = plotlib.fig("col", 1.9)
cells = np.array([[both, only_stack], [only_forest, neither]])
ax.imshow([[0, 1], [1, 0]], cmap="Blues", vmin=0, vmax=2.5)
for (i, j), n in np.ndenumerate(cells):
    ax.text(j, i, str(n), ha="center", va="center", fontweight="bold" if i != j else "normal")
ax.set_xticks([0, 1], ["forest right", "forest wrong"])
ax.set_yticks([0, 1], ["stacking right", "stacking wrong"], rotation=90, va="center")
ax.tick_params(length=0)
ax.grid(False)
for side in ax.spines.values():
    side.set_visible(False)
plotlib.save(fg, "e8_table")

## Every rival on every table

Six rivals mean six tests per table, and some will pass by luck. Holm's correction keeps the chance of any false claim per table at 5 %. The table prints McNemar next to the permutation test on MCC: they usually agree, because they read the same disagreements.

In [ ]:
slide("Stacking Against Six Rivals: Classification")


def compare(dataset):
    task = TABLES[dataset][1]
    y, stack = predictions(dataset, "stacking")
    rows = []
    for rival in RIVALS:
        other = predictions(dataset, rival)[1]
        gain, perm = metrics.permutation_test(task, y, stack, other)
        low, high = metrics.bootstrap_ci(task, y, stack, other)
        row = {"dataset": dataset, "rival": rival, "gain": gain, "low": low, "high": high, "perm": perm}
        if task == "c":
            row |= dict(zip(("only stack", "only rival", "mcnemar"), metrics.mcnemar(y, stack, other)))
        rows.append(row)
    frame = pl.DataFrame(rows).with_columns(pl.Series("perm holm", metrics.holm([r["perm"] for r in rows])))
    return (
        frame.with_columns(pl.Series("mcnemar holm", metrics.holm([r["mcnemar"] for r in rows])))
        if task == "c"
        else frame
    )


tests_c = pl.concat([compare(d) for d in CLASS_SETS])
show(tests_c.select("dataset", "rival", "only stack", "only rival", "gain", "mcnemar holm", "perm holm"))
agree = tests_c.filter((pl.col("mcnemar holm") < ALPHA) == (pl.col("perm holm") < ALPHA)).height
print(f"McNemar and the permutation test agree on {agree} of {tests_c.height} comparisons")
print(
    f"stacking significantly better: McNemar {tests_c.filter(pl.col('mcnemar holm') < ALPHA, pl.col('gain') > 0).height}, permutation {tests_c.filter(pl.col('perm holm') < ALPHA, pl.col('gain') > 0).height}"
)
print(f"stacking significantly worse: {tests_c.filter(pl.col('perm holm') < ALPHA, pl.col('gain') < 0).height}")
show(
    tests_c.filter((pl.col("mcnemar holm") < ALPHA) != (pl.col("perm holm") < ALPHA)).select(
        "dataset", "rival", "only stack", "only rival", "gain", "low", "high", "mcnemar holm", "perm holm"
    )
)

## One family of tests

McNemar's test is a sign test on the disagreements. Friedman's test is its generalization: on 0/1 outcomes it is Cochran's Q, and with two models it is McNemar again (without the continuity correction). So Friedman is not a rival of McNemar: they are the same idea with different samples.

| Samples | Two models | Many models |
|:---|:---|:---|
| test rows, right or wrong | McNemar | Cochran's Q |
| test rows, loss of each row | sign test, Wilcoxon | Friedman on rows |
| tables, one score per table | sign test, Wilcoxon | Friedman on tables |

The next cell checks the identities on spambase and runs the regression twin, Friedman on the absolute errors of the rows, on California housing.

In [ ]:
slide("One Family of Tests")


def cochran_q(right):
    """Cochran's Q of a 0/1 matrix, one row per test row, one column per model."""
    k, by_model, by_row = right.shape[1], right.sum(0), right.sum(1)
    return (k - 1) * (k * (by_model**2).sum() - by_model.sum() ** 2) / (k * by_row.sum() - (by_row**2).sum())


y, stack = predictions("spambase", "stacking")
right = np.array([predictions("spambase", m)[1] == y for m in ["stacking", *RIVALS]], float).T
only_a, only_b, _ = metrics.mcnemar(y, stack, predictions("spambase", "random forest")[1])
two = right[:, [0, RIVALS.index("random forest") + 1]]
print(
    f"two models: (b - c)^2 / (b + c) = {(only_a - only_b) ** 2 / (only_a + only_b):.3f}, Cochran's Q = {cochran_q(two):.3f}"
)
print(
    f"seven models: Cochran's Q = {cochran_q(right):.1f}, Friedman on the rows = {stats.friedmanchisquare(*right.T).statistic:.1f}, p = {stats.friedmanchisquare(*right.T).pvalue:.1e}"
)
y, _ = predictions("California housing", "stacking")
losses = np.array([np.abs(predictions("California housing", m)[1] - y) for m in ["stacking", *RIVALS]]).T
print(
    f"California housing, Friedman on the absolute errors of {len(y)} rows: chi2 = {stats.friedmanchisquare(*losses.T).statistic:.1f}, p below 1e-300: {stats.friedmanchisquare(*losses.T).pvalue < 1e-300}"
)
mean_rank_rows = stats.rankdata(losses, axis=1).mean(0)
print("mean rank per row:", {m: round(float(r), 2) for m, r in zip(["stacking", *RIVALS], mean_rank_rows)})

## McNemar does not read MCC

McNemar counts errors, and every error weighs the same. MCC does not: on an imbalanced table, an error on the rare class costs more. The table above already holds a real case: when the two tests disagree, it is on German credit, the table with 30 % of one class, and the gain is in MCC. Here is the same effect in pure form. Take one forest on German credit (30 % bad) and move its decision threshold until it makes about as many errors as before. The two versions disagree on many rows, the error count stays, and MCC changes, yet McNemar cannot see it: its answer is p = 1 whatever the rows. Here the change in MCC is itself too small to pass the permutation test, so this shows blindness, not a missed result. The threshold is picked on the test rows only to build this example; never do that in a real comparison.

In [ ]:
slide("McNemar Does Not Read MCC")
X, target = data.german_bad()
tr, te = train_test_split(np.arange(len(target)), test_size=0.3, random_state=SEED, stratify=target)
forest = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X[tr], target[tr])
prob = forest.predict_proba(X[te])[:, 1]
at_half = (prob >= 0.5).astype(int)
candidates = [
    t for t in np.linspace(0.2, 0.5, 61) if abs(np.sum((prob >= t) != target[te]) - np.sum(at_half != target[te])) <= 1
]
moved = (prob >= min(candidates)).astype(int)
a, b, p_mcnemar = metrics.mcnemar(target[te], moved, at_half)
gain, p_perm = metrics.permutation_test("c", target[te], moved, at_half)
low, high = metrics.bootstrap_ci("c", target[te], moved, at_half)
print(f"threshold 0.50: errors {np.sum(at_half != target[te])}, MCC {metrics.task_score('c', target[te], at_half):.3f}")
print(
    f"threshold {min(candidates):.3f}: errors {np.sum(moved != target[te])}, MCC {metrics.task_score('c', target[te], moved):.3f}"
)
print(
    f"disagreements {a} against {b}: McNemar p = {p_mcnemar:.3f}; permutation on MCC p = {p_perm:.3f}, gain {gain:+.3f} [{low:+.3f}, {high:+.3f}]"
)

## Regression: no McNemar

A regression answer is never simply right, so the 2×2 table does not exist. Three replacements work on the paired errors of each row:

* **Sign test:** which model came closer on this row? This is McNemar's binomial test with "right" replaced by "closer". It ignores how much closer.
* **Wilcoxon signed-rank:** the differences of absolute errors, by rank. It uses size, but it does not test MAE.
* **Permutation test on MAE:** swap the two predictions of random rows and recompute MAE. It tests the metric we report.

In [ ]:
slide("Regression: Three Tests")
tests_r = pl.concat([compare(d) for d in REG_SETS])


def row_tests(dataset, rival):
    y, stack = predictions(dataset, "stacking")
    other = predictions(dataset, rival)[1]
    diff = np.abs(other - y) - np.abs(stack - y)
    closer, farther = int(np.sum(diff > 0)), int(np.sum(diff < 0))
    return {
        "dataset": dataset,
        "rival": rival,
        "closer": closer,
        "farther": farther,
        "sign": stats.binomtest(closer, closer + farther, 0.5).pvalue,
        "wilcoxon": stats.wilcoxon(diff).pvalue,
    }


rows_r = pl.DataFrame([row_tests(d, r) for d in REG_SETS for r in RIVALS])
tests_r = tests_r.join(rows_r, on=["dataset", "rival"]).with_columns(
    pl.Series(
        "sign holm",
        np.concatenate([metrics.holm(rows_r.filter(pl.col("dataset") == d)["sign"].to_numpy()) for d in REG_SETS]),
    ),
    pl.Series(
        "wilcoxon holm",
        np.concatenate([metrics.holm(rows_r.filter(pl.col("dataset") == d)["wilcoxon"].to_numpy()) for d in REG_SETS]),
    ),
)
show(
    tests_r.select(
        "dataset", "rival", "gain", "low", "high", "closer", "farther", "sign holm", "wilcoxon holm", "perm holm"
    )
)
for name in ("sign holm", "wilcoxon holm", "perm holm"):
    print(
        f"{name}: stacking significantly better on {tests_r.filter(pl.col(name) < ALPHA, pl.col('gain') > 0).height} of {tests_r.height}, worse on {tests_r.filter(pl.col(name) < ALPHA, pl.col('gain') < 0).height}"
    )

The three tests answer three questions: closer more often, smaller errors by rank, smaller mean error. A significant result on one is not a result on the others. The next cell prints two cases: the verdicts that differ, and a gain that is significant but tiny.

In [ ]:
slide("Three Tests, Three Questions")
flipped = tests_r.filter((pl.col("closer") > pl.col("farther")) != (pl.col("gain") > 0))
print(
    f"{flipped.height} of {tests_r.height} comparisons: the model closer on most rows has the worse MAE (none significant: {flipped.filter(pl.col('sign holm') < ALPHA).height == 0})"
)
split = tests_r.filter((pl.col("sign holm") < ALPHA) != (pl.col("perm holm") < ALPHA))
show(
    split.select(
        "dataset", "rival", "gain", "low", "high", "closer", "farther", "sign holm", "wilcoxon holm", "perm holm"
    )
)
tiny = tests_r.filter(pl.col("dataset") == "California housing", pl.col("rival") == "HistGB")
print(
    f"California housing, stacking against HistGB: gain {tiny['gain'][0]:.4f} [{tiny['low'][0]:.4f}, {tiny['high'][0]:.4f}] on a MAE of {score_of('California housing', 'HistGB'):.3f}, that is {100 * tiny['gain'][0] / score_of('California housing', 'HistGB'):.2f} %, Holm p = {tiny['perm holm'][0]:.3f}"
)

In [ ]:
# plot e8_grid
sets = [*CLASS_SETS, *REG_SETS]
tests = pl.concat(
    [tests_c.select("dataset", "rival", "gain", "perm holm"), tests_r.select("dataset", "rival", "gain", "perm holm")]
)
state = np.zeros((len(sets), len(RIVALS)))
fg, ax = plotlib.fig("full", 2.45)
for d, r, gain, p in tests.iter_rows():
    i, j = sets.index(d), RIVALS.index(r)
    state[i, j] = np.sign(gain) * (p < ALPHA)
    base = score_of(d, r)
    ax.text(j, i, f"{100 * gain / abs(base):+.0f}%", ha="center", va="center", fontsize=6)
ax.imshow(
    state,
    cmap=plt.matplotlib.colors.ListedColormap([plotlib.RED, "#e6e6e6", plotlib.GREEN]),
    vmin=-1,
    vmax=1,
    alpha=0.65,
    aspect="auto",
)
ax.axhline(len(CLASS_SETS) - 0.5, color="k", lw=0.8)
ax.set_xticks(range(len(RIVALS)), RIVALS)
ax.set_yticks(range(len(sets)), sets)
ax.xaxis.tick_top()
ax.tick_params(length=0)
ax.grid(False)
for side in ax.spines.values():
    side.set_visible(False)
plotlib.save(fg, "e8_grid")
slide("Which Gains Survive?")
print(
    f"stacking significantly better on {tests.filter(pl.col('perm holm') < ALPHA, pl.col('gain') > 0).height} of {tests.height}, worse on {tests.filter(pl.col('perm holm') < ALPHA, pl.col('gain') < 0).height}"
)
print({r: tests.filter(pl.col("rival") == r, pl.col("perm holm") < ALPHA, pl.col("gain") > 0).height for r in RIVALS})

In [ ]:
# plot e8_interval
strong = tests_r.filter(pl.col("rival") == "HistGB")
fg, ax = plotlib.fig("full", 1.55)
base = np.array([score_of(d, "HistGB") for d in REG_SETS])
gain, low, high = (100 * strong[c].to_numpy() / base for c in ("gain", "low", "high"))
ypos = np.arange(len(REG_SETS))[::-1]
ax.errorbar(gain, ypos, xerr=[gain - low, high - gain], fmt="o", ms=3, color=plotlib.BLUE, capsize=2)
ax.axvline(0, color=plotlib.GREY, ls=":", lw=0.8)
ax.set_yticks(ypos, REG_SETS)
ax.set_xlabel("gain of stacking over HistGB, % of MAE, with 95 % interval")
ax.grid(False)
plotlib.save(fg, "e8_interval")

## Cross-validation: folds are not independent

With repeated CV it is tempting to run a paired t-test on the fold scores. Those scores share training rows, so they are correlated and the test is too sure of itself. Nadeau and Bengio's correction widens the variance by the ratio of test to training rows.

A simulation shows it with known truth. Two linear classifiers see different blocks of equally useful features, so at `advantage` 0 neither is better, and every rejection is a false alarm. With a positive advantage model A really is better, and rejections are correct. All tests use 300 rows at the 5 % level.

In [ ]:
slide("The Plain t-Test Cries Wolf")
BLOCK = 4
N_SIMS = 400
ADVANTAGES = (0.0, 0.1, 0.2, 0.3, 0.4)


def fit_errors(Z, y, tr, te):
    """Errors on te of a least-squares classifier on the columns Z."""
    design = np.c_[np.ones(len(tr)), Z[tr]]
    w = np.linalg.lstsq(design, 2.0 * y[tr] - 1.0, rcond=None)[0]
    return (np.c_[np.ones(len(te)), Z[te]] @ w > 0) != y[te]


def one_simulation(advantage, seed, n=300, folds=10, repeats=10):
    rng = np.random.default_rng([seed, int(advantage * 100)])
    A, B = rng.normal(size=(n, BLOCK)), rng.normal(size=(n, BLOCK))
    y = ((1 + advantage) * A.sum(1) + B.sum(1) + 2.0 * rng.normal(size=n) > 0).astype(int)
    order = rng.permutation(n)
    tr, te = order[: int(0.7 * n)], order[int(0.7 * n) :]
    wrong_a, wrong_b = fit_errors(A, y, tr, te), fit_errors(B, y, tr, te)
    p_mcnemar = metrics.mcnemar(y[te], np.where(wrong_a, 1 - y[te], y[te]), np.where(wrong_b, 1 - y[te], y[te]))[2]
    gains = []
    for _ in range(repeats):
        for te in np.array_split(rng.permutation(n), folds):
            tr = np.setdiff1d(np.arange(n), te)
            gains.append(fit_errors(B, y, tr, te).mean() - fit_errors(A, y, tr, te).mean())
    gains = np.array(gains)
    p_plain = stats.ttest_1samp(gains, 0.0).pvalue if gains.std() > 0 else 1.0
    return advantage, p_mcnemar, p_plain, metrics.corrected_ttest(gains, 1 / (folds - 1))


with threadpool_limits(1):
    runs = Parallel(n_jobs=-1, batch_size=8)(delayed(one_simulation)(a, s) for a in ADVANTAGES for s in range(N_SIMS))
sim = pl.DataFrame(
    runs, schema=["advantage", "McNemar, hold-out", "plain t, 10×10 CV", "corrected t, 10×10 CV"], orient="row"
)
tests_sim = ["McNemar, hold-out", "plain t, 10×10 CV", "corrected t, 10×10 CV"]
rates = sim.group_by("advantage", maintain_order=True).agg([(pl.col(t) < ALPHA).mean() for t in tests_sim])
show(rates)

In [ ]:
# plot e8_calibration
fg, ax = plotlib.fig("full", 1.55)
for t, colour, marker in zip(tests_sim, (plotlib.BLUE, plotlib.RED, plotlib.GREEN), "osD"):
    ax.plot(rates["advantage"], rates[t], color=colour, marker=marker, ms=3, label=t)
ax.axhline(ALPHA, color=plotlib.GREY, ls=":", lw=0.8)
ax.set(xlabel="true advantage of model A", ylabel="share of tests that reject", ylim=(0, 1))
ax.legend(frameon=False, fontsize=6)
plotlib.save(fg, "e8_calibration")

## Several tables

With one score per table, the tables are the samples and they are independent. Demšar recommends Wilcoxon for two models and Friedman with Nemenyi for many. Benavoli and colleagues add a warning: the verdict of a mean-rank post-hoc test on two models depends on which other models are in the pool, so paired tests with Holm's correction are safer. Gains are relative, so MCC and MAE tables can share one test. The scores come from the tuned, full comparison of E7.

In [ ]:
slide("Across Tables: Wilcoxon and Holm")
results = pl.read_csv(plotlib.ROOT / ".cache" / "07_tables.csv")
means = results.group_by("dataset", "model").agg(pl.col("score").mean())
models = sorted(set(means["model"]) - {"stacking"})
score = {(d, m): s for d, m, s in means.iter_rows()}
gain_of = lambda d, m: (score[d, "stacking"] - score[d, m]) / abs(score[d, m]) * (1 if TABLES[d][1] == "c" else -1)  # noqa: E731
pvalues = [stats.wilcoxon([gain_of(d, m) for d in TABLES]).pvalue for m in models]
ranks = pl.concat(
    [
        means.filter(pl.col("dataset") == d).with_columns(
            pl.col("score").rank("average", descending=t == "c").alias("rank")
        )
        for d, (_, t) in TABLES.items()
    ]
)
mean_rank = dict(ranks.group_by("model").agg(pl.col("rank").mean()).iter_rows())
k, n_tables = len(models) + 1, len(TABLES)
cd = stats.studentized_range.isf(ALPHA, k, 1e6) / np.sqrt(2) * np.sqrt(k * (k + 1) / (6 * n_tables))
across = pl.DataFrame(
    {
        "model": models,
        "median gain %": [100 * float(np.median([gain_of(d, m) for d in TABLES])) for m in models],
        "tables beaten": [sum(gain_of(d, m) > 0 for d in TABLES) for m in models],
        "wilcoxon": pvalues,
        "holm": metrics.holm(pvalues),
        "Nemenyi": [mean_rank[m] - mean_rank["stacking"] > cd for m in models],
    }
).sort("holm")
show(across, 4)
print(
    f"Nemenyi distance {cd:.2f}: behind stacking by Nemenyi {across['Nemenyi'].sum()}, by Wilcoxon with Holm {across.filter(pl.col('holm') < ALPHA).height} of {len(models)}"
)
smallest = 2 / 2**n_tables
pairs = k * (k - 1) // 2
print(
    f"smallest possible two-sided Wilcoxon p with {n_tables} tables: {smallest:.4f}; Holm on all {pairs} pairs multiplies it by {pairs}: {smallest * pairs:.3f}"
)

Read this table with care. Wilcoxon with Holm finds 11 models behind stacking, Nemenyi 6, and they do not test the same family. Holm here corrects for 15 comparisons with one chosen reference, and that reference was chosen after E7 showed it on top. Choosing the winner first is a selection the correction does not pay for. Nemenyi covers all pairs, so it does pay, and that is part of why it finds less. Holm on all 120 pairs cannot reject anything with 11 tables, as the last line shows: the smallest p-value is too large. The honest reading: more tables, or a reference fixed in advance.

A p-value cannot say that two models are equal, and with enough tables it flags gains too small to matter. The Bayesian signed-rank test of Benavoli and colleagues answers a direct question: given these tables, what is the probability that stacking is practically better, practically equal, or worse? "Practically equal" is a region we choose: here, a gain below 1 %. `metrics.bayes_signed_rank` matches the `baycomp` library.

In [ ]:
slide("Across Tables: Bayesian Signed-Rank")
ROPE = 0.01
bayes = pl.DataFrame(
    [
        dict(
            zip(
                ("model", "P(worse)", "P(equal)", "P(better)"),
                (m, *metrics.bayes_signed_rank([gain_of(d, m) for d in TABLES], ROPE)),
            )
        )
        for m in models
    ]
).sort("P(better)", descending=True)
show(bayes)
print(f"P(better) above 0.95 for {bayes.filter(pl.col('P(better)') > 0.95).height} of {len(models)}")

Within one table, the fold scores of E7 feed the corrected t-test: stacking against each of the 15 others, with Holm. Five folds give only four degrees of freedom, and the next cell shows what that costs.

In [ ]:
slide("Small Tables Say Little")
wide = results.pivot(on="model", index=["dataset", "seed"], values="score")
found = []
for d, (_, task) in TABLES.items():
    here = wide.filter(pl.col("dataset") == d)
    sign = 1 if task == "c" else -1
    ps = [metrics.corrected_ttest(sign * (here["stacking"].to_numpy() - here[m].to_numpy()), 1 / 4) for m in models]
    found.append(
        {
            "dataset": d,
            "best p": min(ps),
            "better with Holm": int(
                np.sum((metrics.holm(ps) < ALPHA) & (np.array([gain_of(d, m) for m in models]) > 0))
            ),
        }
    )
found = pl.DataFrame(found)
show(found, 4)
print(f"tables with no significant gain: {found.filter(pl.col('better with Holm') == 0).height}")

On the four tables with no result the test sees nothing, even where the gain is large. A silent test on a small table says the data are too few; it does not say the models are equal.

## Which test, when

| Question | Test | Task |
|:---|:---|:---|
| Two fitted models, one test set, error rate | McNemar | classification |
| Two fitted models, one test set, any metric | paired permutation, bootstrap interval | both |
| Two learners on one table, repeated CV | corrected t-test | both |
| Many models on many tables | Wilcoxon with Holm; Friedman first; Bayesian signed-rank | both |

McNemar and Friedman do not compete: they answer questions at different levels, and they are one family of tests. Order of preference for this class: the permutation test with an interval, because it tests MCC and MAE themselves. A p-value says a gain is probably not luck; the interval says whether it is worth having. Time series need Diebold-Mariano, because their rows are not exchangeable. 

These tests compare fitted models on one test set, so they cover the noise of the test rows and not the noise of the training rows. Only the repeated-CV row covers both. Bates, Hastie and Tibshirani add that cross-validation estimates the average error over training sets, not the error of the model at hand, and that naive intervals can undercover; Bayle and colleagues give intervals and tests for k-fold error with exact asymptotic coverage.